# Interactive Training and Learning

This notebook demonstrates the acetate learning workflow interactively.
It can retrain the LGP surrogates, rerun posterior sampling, export
validation-ready parameter samples, and generate a couple of posterior
plots.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from bff.bayes.results import InferenceResults
from bff.plotting import plot_corner, plot_marginals
from bff.workflows.learn import main as learn_workflow
from bff.workflows.train import main as train_workflow

ROOT = Path('.').resolve()
TRAIN_CONFIG = ROOT.parent / '05-train-lgp' / 'config.yaml'
LEARN_CONFIG = ROOT / 'config.yaml'
SPECS = ROOT.parent / '03-training-trjs' / 'trainset' / 'specs.yaml'

ROOT, TRAIN_CONFIG, LEARN_CONFIG, SPECS

## 1. Retrain the surrogate models

Re-run this cell if you changed any dataset or training settings.

In [ ]:
train_workflow(TRAIN_CONFIG)

## 2. Run posterior inference

This uses the trained `.lgp` files from stage `05-train-lgp`.

In [ ]:
learn_workflow(LEARN_CONFIG)

## 3. Load, prepare, and export posterior samples

The exported `posterior-samples.yaml` can be used directly by the
`08-validate` stage.

In [ ]:
results = InferenceResults.load(
    posterior=ROOT / 'posterior.pt',
    priors=ROOT / 'priors.pt',
    specs=SPECS,
)
results.prepare_samples()
results.sample_parameters(
    n_samples=10,
    fn_out=ROOT / 'posterior-samples.yaml',
    overwrite=True,
)
results.map_estimates

## 4. Plot the posterior

The figures are saved next to the notebook and displayed inline.

In [ ]:
plot_corner(results, fn_out=ROOT / 'posterior-corner.png')
plot_marginals(results, SPECS, fn_out=ROOT / 'posterior-marginals.png')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, name in zip(
    axes,
    ['posterior-corner.png', 'posterior-marginals.png'],
):
    ax.imshow(plt.imread(ROOT / name))
    ax.axis('off')
plt.tight_layout()
plt.show()